# MAGFiLO MaskDINO (detectron2, ResNet-50)

Add Data → `filament-segmentation-2026`, GPU on.

**Перед запуском:** закоммить и запушь ветку `feature/mask-dino-dev` (ноутбук клонирует её с GitHub).

Порядок: установка → `prepare_data` → smoke → полное обучение.

detectron2 ставим в **системный Python** (не в uv-venv): только там предустановлен GPU-торч Kaggle.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN", "DAGSHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

os.chdir("/kaggle/working")
REPO = Path("/kaggle/working/filament-segmentation-2026")

# чистая копия: убираем возможный недоклонированный репозиторий (остаётся .git)
if REPO.exists():
    shutil.rmtree(REPO)

url = os.environ.get("GIT_URL") or "https://github.com/d1maal/filament-segmentation-2026"
branch = os.environ.get("GIT_BRANCH") or "feature/mask-dino-dev"
token = os.environ.get("GITHUB_TOKEN", "")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)

# subprocess: check_call упадёт при ошибке и не засветит токен в выводе
try:
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", branch, url, str(REPO)]
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f"git clone failed (exit {exc.returncode}); проверь GIT_URL/GITHUB_TOKEN и ветку '{branch}'"
    ) from None

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("repo:", REPO)

import torch
print(
    "torch:", torch.__version__,
    "| cuda:", torch.version.cuda,
    "| gpu:", torch.cuda.is_available(),
)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!pip install -q git+https://github.com/facebookresearch/detectron2.git

In [ ]:
import os

# CUDA-ядро MSDeformAttn
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
ops = "src/models/mask_dino_dtn/maskdino/modeling/pixel_decoder/ops"

In [ ]:
!cd {ops} && bash make.sh

In [ ]:
import MultiScaleDeformableAttention
import detectron2
print("detectron2 + MultiScaleDeformableAttention OK")

In [ ]:
!python -m src.models.mask_dino_dtn.prepare_data

In [ ]:
!python run.py check

In [ ]:
!python run.py mask_dino_dtn train --smoke --run-name smoke

In [ ]:
# !python run.py mask_dino_dtn train --size 512 --epochs 8 --run-name mask_dino_dtn_run